In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# project overview
**This project analyzes TV Marketing (Advertising) data to understand how advertising spend across different channels impacts product Sales.

The project is treated as a real-world business problem, focusing not only on prediction accuracy but also on interpretability, insights, and decision-making support.**

# import libraries


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# load datasets # 

In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/input/advertising-dataset/advertising.csv')
df.head()


# EDA & visualization

In [ ]:
df.describe().T

In [ ]:
df.info()

**no missing values & datatype is float**

In [ ]:
df.isna().sum().plot(kind='bar')

In [ ]:
df.isnull().sum()

**there isnot any missing & null values**

In [ ]:
sns.pairplot(df)
plt.show()


In [ ]:
sns.heatmap(df.corr(),annot =True)

**we find that a tv has astrong relation with sales**

In [ ]:
fig, axs = plt.subplots(4, figsize = (7,11))
plt1 = sns.boxplot(df['TV'], ax = axs[0])
plt2 = sns.boxplot(df['Newspaper'], ax = axs[1])
plt3 = sns.boxplot(df['Radio'], ax = axs[2])
plt3 = sns.boxplot(df['Sales'], ax = axs[3])


plt.tight_layout()

# Features and Target

In [ ]:
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

In [ ]:
print(X.shape)
print(y.shape)

# split data 

In [ ]:
from sklearn.model_selection import train_test_split 
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# try linear regression model 

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train,y_train)


# Coefficients results

In [ ]:
# Print the intercept and coefficients
print(lr.intercept_)
print(lr.coef_)

# prediction

In [ ]:
y_pred = lr.predict(X_test)
y_pred

In [ ]:
y_test

# compare actual output vs predicted

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linestyle='--')

plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales - Linear Regression")
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(y_test.values, label='Actual')
plt.plot(y_pred, label='Predicted')
plt.legend()
plt.title("Actual vs Predicted Sales")
plt.show()


In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals)
plt.axhline(0, linestyle='--')

plt.xlabel("Predicted Sales")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.show()


# evaluate the mode

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
print("Linear Regression R2:", r2_score(y_test, y_pred))
print("Linear Regression MAE:", mean_absolute_error(y_test, y_pred))
print("Linear Regression MSE:", mean_squared_error(y_test, y_pred))


In [ ]:
import matplotlib.pyplot as plt
plt.scatter(y_test,y_pred)
plt.xlabel('Y Test')
plt.ylabel('Predicted Y')

# using regularization to get best score (ridge & lasso)

In [ ]:
from sklearn.linear_model import Ridge , Lasso

In [ ]:
lasso = Lasso(alpha=0.01)
lasso.fit(X_train, y_train)

y_pred_lasso = lasso.predict(X_test)
print("Lasso R2:", r2_score(y_test, y_pred_lasso))


In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)
print("Ridge R2:", r2_score(y_test, y_pred_ridge))


#  Coefficients compare 

In [ ]:
coef_df = pd.DataFrame({
    'Linear': lr.coef_,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_
}, index=X.columns)

coef_df.plot(kind='bar', title='Feature Importance Comparison')
plt.show()

# Hyperparameter Tuning ( choose best Alpha) & balance bias and variance 

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid_ridge = {'alpha': [0.01, 0.1, 1, 10, 100]}

grid_ridge = GridSearchCV(
    Ridge(),
    param_grid_ridge,
    cv=5,
    scoring='r2'
)

grid_ridge.fit(X_train, y_train)

print("Best Ridge Alpha:", grid_ridge.best_params_)


**BEST VALUE FOR ALPHA IN RIDGE IS 100 **

# USE GRIDSEARCHCV FOR FIND BEST VALUE FOR ALPHA IN LASSO & RIDGE

In [ ]:
param_grid_lasso = {'alpha': [0.001, 0.01, 0.1, 1]}

grid_lasso = GridSearchCV(
    Lasso(max_iter=5000),
    param_grid_lasso,
    cv=5,
    scoring='r2'
)

grid_lasso.fit(X_train, y_train)

print("Best Lasso Alpha:", grid_lasso.best_params_)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(y_test.values, label='Actual')
plt.plot(y_pred_lasso, label='Predicted')
plt.legend()
plt.title("Actual vs Predicted Sales")
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(y_test.values, label='Actual')
plt.plot(y_pred_ridge, label='Predicted')
plt.legend()
plt.title("Actual vs Predicted Sales")
plt.show()

**BEST VALUE FOR ALPHA IN LASSO IS 0.1**

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
from sklearn.pipeline import Pipeline

# use PolynomialFeatures

In [ ]:
poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr', LinearRegression())
])

poly_model.fit(X_train, y_train)

y_pred_poly = poly_model.predict(X_test)

print("Polynomial Regression R2:", r2_score(y_test, y_pred_poly))


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(y_test.values, label='Actual')
plt.plot(y_pred_poly, label='Predicted')
plt.legend()
plt.title("Actual vs Predicted Sales")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_poly)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linestyle='--')

plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales - Linear Regression")
plt.show()


# final insights 

In [ ]:
print("""
Final Insights:
- TV advertising has the strongest impact on Sales
- Ridge improves model stability
- Lasso highlights unimportant features
- Polynomial Regression slightly improves performance but reduces interpretability
""")
